# 🎙️ Whisper — Tunisian Arabic ASR Benchmark

**Model:** `openai/whisper-large-v3`  
**Config key:** `whisper_large_v3`

## 1 · Install & Imports

In [1]:
!pip install -q transformers datasets torchaudio evaluate jiwer pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp -r '/content/drive/My Drive/asr_benchmark' /content/asr_benchmark
import sys
sys.path.insert(0, '/content/asr_benchmark')

In [4]:
from benchmark_utils import (
    load_config, get_device, print_gpu_info, setup_output_dir,
    load_benchmark, split_benchmark,
    compute_metrics, per_sample_wer,
    run_pipeline_inference, build_results_df,
    run_labelled_splits, run_unlabelled_splits,
    display_preview, display_worst, display_bulk_predictions,
    audio_inspector, display_summary, plot_wer_cer,
)
import numpy as np
import torch
import time
from tqdm.auto import tqdm
from datasets import Audio as HFAudio

cfg = load_config('/content/asr_benchmark/config.yaml')
TARGET_SR    = cfg['evaluation']['target_sr']
TOP_N_WORST  = cfg['evaluation']['top_n_worst']
PREVIEW_ROWS = cfg['evaluation']['preview_rows']

## 2 · GPU Check

In [5]:
device = get_device()
print_gpu_info()
torch_dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'torch_dtype : {torch_dtype}')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB total  |  15.6 GB free
torch_dtype : torch.float16


## 3 · Load Model & Pipeline

### 3.1 Load processor and model

Whisper is a **seq2seq encoder-decoder** model. We force Arabic generation
via `language` and `task` in `generate_kwargs`.

> **Fix — duplicate logits processors:** We pass `language` and `task` only
> through `generate_kwargs` in the pipeline constructor, and explicitly set
> them on the generation config **before** calling `from_pretrained` so that
> `SuppressTokensLogitsProcessor` and `SuppressTokensAtBeginLogitsProcessor`
> are not duplicated at call time.

In [6]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline

mcfg       = cfg['models']['whisper_large_v3']
MODEL_ID   = mcfg['model_id']
BATCH_SIZE = mcfg['batch_size']
OUTPUT_DIR = setup_output_dir(cfg, 'whisper_large_v3')

print(f'Loading {MODEL_ID} ...')
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
)

# Ensure forced_decoder_ids is cleared; the pipeline sets it via generate_kwargs
model.generation_config.forced_decoder_ids = None

model.to(device)
model.eval()

pipe = pipeline(
    'automatic-speech-recognition',
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=mcfg['chunk_length_s'],
    stride_length_s=mcfg['stride_length_s'],
      generate_kwargs={
        'language': mcfg['language'],
        'task':     mcfg['task'],
    },
)
print(f'✓ {MODEL_ID} loaded on {device} ({torch_dtype})')

Loading openai/whisper-large-v3 ...


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ openai/whisper-large-v3 loaded on cuda (torch.float16)


## 4 · Mount Drive & Load Benchmark

In [7]:
benchmark = load_benchmark(cfg)
LABELLED_SPLITS, UNLABELLED_SPLITS = split_benchmark(benchmark)

Split                            Rows  Has transcript  Duration (h)
--------------------------------------------------------------------
  labeled_algerian                400            True        3.175
  labeled_linagora_raw           2380            True        3.187
  labeled_linagora_cs_arabize    2380            True        3.187
  labeled_linagora_cs_keep       2380            True        3.187
  unlabeled_youtube               396           False        3.274
Labelled splits   : ['labeled_algerian', 'labeled_linagora_raw', 'labeled_linagora_cs_arabize', 'labeled_linagora_cs_keep']
Unlabelled splits : ['unlabeled_youtube']


## 5 · Inference

In [7]:
def infer_fn(ds):
    return run_pipeline_inference(ds, pipe, BATCH_SIZE, TARGET_SR)

all_result_dfs, summary_rows = run_labelled_splits(
    benchmark, LABELLED_SPLITS, infer_fn,
    OUTPUT_DIR, 'whisper_largev3',
    PREVIEW_ROWS, TOP_N_WORST,
)


  Running: labeled_algerian  (400 samples)


Inferring:   0%|          | 0/100 [00:00<?, ?batch/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
You seem to be using the pipeli

  WER              : 1.1153
  CER              : 0.7587
  RTF              : 0.3740
  Total audio      : 3.175 h
  Inference time   : 4275.2 s
  Mean latency/smp : 10.6537 s
  Median latency   : 8.2870 s
  ✓ CSV saved → /content/drive/My Drive/asr_benchmark_results/whisper_largev3_results/labeled_algerian_whisper_largev3.csv

  Running: labeled_linagora_raw  (2380 samples)


Inferring:   0%|          | 0/595 [00:00<?, ?batch/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2372/557137963.py", line 4, in <cell line: 0>
    all_result_dfs, summary_rows = run_labelled_splits(
                                   ^^^^^^^^^^^^^^^^^^^^
  File "/content/asr_benchmark/benchmark_utils.py", line 236, in run_labelled_splits
    result = infer_fn(ds)
             ^^^^^^^^^^^^
  File "/tmp/ipykernel_2372/557137963.py", line 2, in infer_fn
    return run_pipeline_inference(ds, pipe, BATCH_SIZE, TARGET_SR)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/asr_benchmark/benchmark_utils.py", line 151, in run_pipeline_inference
    results = pipe(inputs, batch_size=batch_size)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/automatic_speech_recognition.py

TypeError: object of type 'NoneType' has no len()

## 6 · Per-sample preview

In [ ]:
display_preview(all_result_dfs, PREVIEW_ROWS)

## 7 · Worst predictions

In [ ]:
display_worst(all_result_dfs, TOP_N_WORST)

## 8 · Unlabelled inspection

In [ ]:
unlabelled_result_dfs = run_unlabelled_splits(
    benchmark, UNLABELLED_SPLITS, infer_fn, OUTPUT_DIR, 'whisper_largev3'
)

In [ ]:
audio_inspector(
    benchmark,
    unlabelled_result_dfs,
    UNLABELLED_SPLITS,
    target_sr=TARGET_SR,
)

In [ ]:
display_bulk_predictions(unlabelled_result_dfs)

## 9 · Summary

In [ ]:
summary_df = display_summary(
    summary_rows, OUTPUT_DIR, 'whisper_largev3', 'openai/whisper-large-v3'
)

In [ ]:
if summary_df is not None:
    plot_wer_cer(summary_df, OUTPUT_DIR, 'whisper_largev3', 'openai/whisper-large-v3')